# Deconvolution
Use the selected 210 genes for deconvolution. Try using different age range of samples as reference panels for age prediction.

In [ ]:
import pandas as pd
import numpy as np
from utils.misc import extract_number, mae
from utils.deconvolution import predict_l2
from utils.viz import single_line_plot, single_scatter_plot

Load the data.

In [ ]:
gene_expressions = pd.read_csv("data/train_data.csv", index_col=0)
gene_expressions_mat = gene_expressions.to_numpy()
genenames = np.array(gene_expressions.index.tolist())
samples = gene_expressions.columns.tolist()

# extract ages
ages = np.array([extract_number(timestring) for timestring in samples])
unique_ages=np.unique(ages)

# retain genes that are present in all samples
prevalence = np.mean(gene_expressions_mat > 0, axis=1)
subset_gene_id = np.where(prevalence == 1)[0]
subset_genenames = genenames[subset_gene_id]
gene_expressions = gene_expressions.loc[subset_genenames, :]
gene_expressions_mat = gene_expressions_mat[subset_gene_id, :]

# get log expressions
log_gene_expressions = np.log(gene_expressions)
log_gene_expressions_mat = np.log(gene_expressions_mat)

# transpose count tables to samples by genes
gene_expressions = gene_expressions.T
gene_expressions_mat = gene_expressions_mat.T
log_gene_expressions = log_gene_expressions.T
log_gene_expressions_mat = log_gene_expressions_mat.T

# get rankings of samples for each gene expression
# gene_expressions_rank = log_gene_expressions.rank()

In [ ]:
# predictor genes
predictor_genes_df = pd.read_csv("gene_plots/markergene_6_23.csv", index_col=0)
predictor_genes = predictor_genes_df.index.tolist()

In [ ]:
predictor_gene_expressions = gene_expressions.loc[:, predictor_genes].to_numpy()
sqrt_predictor_gene_expressions = np.sqrt(predictor_gene_expressions)
log_predictor_gene_expressions = np.log(predictor_gene_expressions)

# Leave one out prediction performance within the samples with known agese

I tried using samples of all ages (2-42) or only using samples of age 2-23 as reference panel for deconvlution. First use samples of all ages.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
# first use all the samples
nsamples = len(ages)
predicted_ages = np.zeros(nsamples)
weights = np.zeros((nsamples, nsamples))


In [ ]:
for j in range(nsamples):

    train_expressions = np.delete(log_predictor_gene_expressions, j, axis=0)
    scaler.fit(train_expressions)
    train_expressions = scaler.transform(train_expressions)
    target_expressions = log_predictor_gene_expressions[j, :].reshape(1, -1)
    target_expressions = scaler.transform(target_expressions).flatten()

    train_ages = np.delete(ages, j)
    estimated_weights, predicted_age = predict_l2(X=train_expressions, Y=target_expressions,
                                                  labels=train_ages)
    estimated_weights = np.insert(estimated_weights, j, 0)
    weights[j, :] = estimated_weights
    predicted_ages[j] = predicted_age


In [ ]:
prediction_2_42 = pd.DataFrame({"Truth": ages, "Predicted": predicted_ages})

In [ ]:
prediction_2_42.to_csv("deconvolution/tables/predictions_2_42.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import os

In [ ]:
fig = single_scatter_plot(ymat=predicted_ages.reshape(1, -1),
                                          xticks=ages, xticknames=ages.astype(str),
                                         xname="True Age (Month)", yname="Predicted Age (Month)", title="Prediction with Samples Aged 2-42",
                                         size=(6, 4), diag_line=True)
folder = "deconvolution/plots/scatterplots/reference_2_42"
filename = f"Prediction_2_42.pdf"
fig.savefig(os.path.join(folder, filename), bbox_inches="tight")
plt.close(fig)

In [ ]:
indices_names = [f"Sample{j}_{ages[j]}" for j in range(nsamples)]
weights_2_42_df = pd.DataFrame(weights, index=indices_names, columns=indices_names)

In [ ]:
weights_2_42_df.to_csv("deconvolution/tables/weights_2_42.csv")

In [ ]:
for j in range(nsamples):

    true_age = ages[j]

    selected_weights = np.delete(weights[j, :], j)

    train_ages = np.delete(ages, j)
    unique_ages = np.unique(train_ages)
    ids_identical = np.where(train_ages == true_age)[0]
    correct_regions = [np.min(ids_identical)+1-0.5, np.max(ids_identical)+1+0.5]
    xticks = np.zeros(len(unique_ages))

    # only have one ticks for each unique age
    for index, unique_age in enumerate(unique_ages):
        ids = np.where(train_ages == unique_age)[0]
        xticks[index] = np.min(ids)+1

    xnames = unique_ages.astype(str)

    title = f"True Age: {true_age}|Predicted Age: {predicted_ages[j]:.2f}"
    fig = single_line_plot(ymat=selected_weights.reshape(1, -1), xmat=np.arange(1, nsamples).reshape(1, -1),
                       xticks=xticks, xticknames=xnames, xname="Age (Months)",
                       yname="Weight",
                       vertical_lines=correct_regions,
                       title=title, size=(7, 4))

    folder = "deconvolution/plots/weights/reference_2_42"
    filename = f"sample_{j+1}_age_{true_age}.pdf"

    fig.savefig(os.path.join(folder, filename), bbox_inches="tight")

    plt.close(fig)

Then I fit deconvolution based on a reference panel with only samples from age 2-23.

In [ ]:
# first use all the samples
nsamples = len(ages)
predicted_ages = np.zeros(nsamples)
weights = np.zeros((nsamples, nsamples))

mask = ages <= 23
weights = weights[:, mask]

In [ ]:
for j in range(nsamples):

    target_expressions = log_predictor_gene_expressions[j, :].reshape(1, -1)
    target_age = ages[j]

    train_expressions = np.delete(log_predictor_gene_expressions, j, axis=0)
    train_ages = np.delete(ages, j)
    # is the target age between 2-23?
    train_expressions = train_expressions[train_ages <= 23, :]
    train_ages = train_ages[train_ages <= 23]

    # standardize
    scaler.fit(train_expressions)
    train_expressions = scaler.transform(train_expressions)
    target_expressions = scaler.transform(target_expressions).flatten()

    estimated_weights, predicted_age = predict_l2(X=train_expressions, Y=target_expressions,
                                                  labels=train_ages)
    if target_age <= 23:
        estimated_weights = np.insert(estimated_weights, j, 0)
    weights[j, :] = estimated_weights
    predicted_ages[j] = predicted_age

In [ ]:
prediction_2_23 = pd.DataFrame({"Truth": ages, "Predicted": predicted_ages})

In [ ]:
prediction_2_23.to_csv("deconvolution/tables/predictions_2_23.csv", index=False)

In [ ]:
fig = single_scatter_plot(ymat=predicted_ages.reshape(1, -1),
                                          xticks=ages, xticknames=ages.astype(str),
                                         xname="True Age (Month)", yname="Predicted Age (Month)", title="Prediction with Samples Aged 2-23",
                                         size=(6, 4), diag_line=True)
folder = "deconvolution/plots/scatterplots/reference_2_23"
filename = f"Prediction_2_23.pdf"
fig.savefig(os.path.join(folder, filename), bbox_inches="tight")
plt.close(fig)

In [ ]:
ages_subset = ages[ages <= 23]
for j in range(nsamples):

    true_age = ages[j]
    if true_age <= 23:
        selected_weights = np.delete(weights[j, :], j)
        train_ages = np.delete(ages_subset, j)
    else:
        selected_weights = weights[j, :]
        train_ages = ages_subset


    unique_ages = np.unique(train_ages)
    correct_regions = None
    if true_age <= 23:
        ids_identical = np.where(train_ages == true_age)[0]
        correct_regions = [np.min(ids_identical)+1-0.5, np.max(ids_identical)+1+0.5]
    xticks = np.zeros(len(unique_ages))

    # only have one ticks for each unique age
    for index, unique_age in enumerate(unique_ages):
        ids = np.where(train_ages == unique_age)[0]
        xticks[index] = np.min(ids)+1

    xnames = unique_ages.astype(str)

    title = f"True Age: {true_age}|Predicted Age: {predicted_ages[j]:.2f}"
    fig = single_line_plot(ymat=selected_weights.reshape(1, -1), xmat=np.arange(1, len(train_ages)+1).reshape(1, -1),
                       xticks=xticks, xticknames=xnames, xname="Age (Months)",
                       yname="Weight",
                       vertical_lines=correct_regions,
                       title=title, size=(7, 4))

    folder = "deconvolution/plots/weights/reference_2_23"
    filename = f"sample_{j+1}_age_{true_age}.pdf"

    fig.savefig(os.path.join(folder, filename), bbox_inches="tight")

    plt.close(fig)

In [ ]:
weights_2_23_df = pd.DataFrame(weights, index=indices_names, columns=indices_names[0:40])

In [ ]:
weights_2_23_df.to_csv("deconvolution/tables/weights_2_23.csv", index=True)

# Prediction of samples with unknown ages

In [ ]:
gene_expressions_test = pd.read_csv("data/test_data.csv", index_col=0)
gene_expressions_test = gene_expressions_test.loc[predictor_genes, :].T

In [ ]:
n_test = gene_expressions_test.shape[0]

In [ ]:
train_raw_expressions = predictor_gene_expressions[ages <= 23, :]
train_log_expressions = log_predictor_gene_expressions[ages <= 23, :]
train_ages = ages[ages <= 23]

In [ ]:
predicted_ages = np.zeros(len(gene_expressions_test))
weights = np.zeros((len(gene_expressions_test), train_raw_expressions.shape[0]))

In [ ]:
scaler.fit(train_log_expressions)
train_log_expressions_scaled = scaler.transform(train_log_expressions)

In [ ]:
for j in range(n_test):

    target_expressions = gene_expressions_test.iloc[j, :].to_numpy()
    if np.any(target_expressions == 0):
        locations = np.where(target_expressions == 0)[0]
        target_expressions[locations] = np.min(train_raw_expressions[:, locations], axis=0)/2

    log_target_expressions = np.log(target_expressions)
    # standardize
    log_target_expressions_scaled = scaler.transform(log_target_expressions.reshape(1, -1)).flatten()

    estimated_weights, predicted_age = predict_l2(X=train_log_expressions_scaled,
                                                  Y=log_target_expressions_scaled,
                                                  labels=train_ages)

    weights[j, :] = estimated_weights
    predicted_ages[j] = predicted_age

In [ ]:
test_prediction_df = pd.DataFrame({"Gene": gene_expressions_test.index.tolist(),
                                   "Predicted":predicted_ages})

In [ ]:
test_prediction_df.to_csv("deconvolution/tables/test_predictions.csv", index=False)

In [ ]:
weights_test_df = pd.DataFrame(weights, index=gene_expressions_test.index.tolist(),
                               columns=indices_names[0:40])

In [ ]:
weights_test_df.to_csv("deconvolution/tables/weights_test.csv")

In [ ]:
for j in range(n_test):

    selected_weights = weights[j, :]

    sample_name = gene_expressions_test.index[j]

    unique_ages = np.unique(train_ages)
    correct_regions = None
    xticks = np.zeros(len(unique_ages))

    # only have one ticks for each unique age
    for index, unique_age in enumerate(unique_ages):
        ids = np.where(train_ages == unique_age)[0]
        xticks[index] = np.min(ids)+1

    xnames = unique_ages.astype(str)

    title = f"{sample_name}|Predicted Age: {predicted_ages[j]:.2f}"
    fig = single_line_plot(ymat=selected_weights.reshape(1, -1),
                           xmat=np.arange(1, len(train_ages)+1).reshape(1, -1),
                       xticks=xticks, xticknames=xnames, xname="Age (Months)",
                       yname="Weight",
                       title=title, size=(7, 4))

    folder = "deconvolution/plots/weights/test"
    filename = f"{sample_name}.pdf"

    fig.savefig(os.path.join(folder, filename), bbox_inches="tight")

    plt.close(fig)